
# YOLOv8n Baseline Evaluation on Roboflow Dataset V4

## Purpose
This notebook evaluates the saved baseline YOLOv8n model on the cleaned Dataset V4 test split. It uses the shared dataset preparation code to keep the evaluation split consistent with the baseline training workflow.

## Methodology
1. Recreate the same cleaned dataset using dataset_preparation.py.
2. Compare the split with the preparation record from training.
3. Load yolov8n_baseline/weights/best.pt.
4. Evaluate precision, recall, mAP50 and mAP50–95 on the test split.

# Input and Output
**Input:** dataset/pork_rasher_v4.zip and model_runs/yolov8n_baseline/weights/best.pt.<br>
**Preparation record:** dataset/manifests/v4_baseline_preparation.json.<br>
**Output:** test metrics and Ultralytics evaluation plots.<br>

## Limitations
The cleaning method uses JPG filename prefixes before `.rf.` to identify augmented sources. Copies with different identifiers may not be detected. The V4 test split has no labelled wrinkle instances, so it cannot establish wrinkle-specific recall or average precision. Matching the preparation record does not confirm the training history of previously saved weights.

In [9]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
from pathlib import Path
import sys

# Base folder for the dataset, trained weights and preparation records.
PROJECT_ROOT = Path("/content/drive/MyDrive/MMA3001_v4_Project")
ZIP_PATH = PROJECT_ROOT / "dataset" / "pork_rasher_v4.zip"
PREPARED_DIR = Path("/content/pork_rasher_dataset_clean")
TRAINING_MANIFEST = PROJECT_ROOT / "dataset" / "manifests" / "v4_baseline_preparation.json"
RUNS_DIR = PROJECT_ROOT / "model_runs"
BASELINE_WEIGHTS = RUNS_DIR / "yolov8n_baseline" / "weights" / "best.pt"

# Locate the shared dataset preparation module.
module_candidates = [
    Path.cwd() / "src",
    Path("/content/src"),
    Path("/content"),
    PROJECT_ROOT / "src",
]
# Include source folders from local GitHub clones.
module_candidates.extend(sorted(Path("/content").glob("*/src")))

module_dir = next((folder for folder in module_candidates
                   if (folder / "dataset_preparation.py").is_file()), None)
if module_dir is None:
    raise FileNotFoundError(
        "dataset_preparation.py not found. Place it in "
        "MyDrive/MMA3001_v4_Project/src/ or /content/src/, "
        "or clone the GitHub repository into /content/."
    )
sys.path.insert(0, str(module_dir))
from dataset_preparation import (  # noqa: E402
    prepare_dataset, print_audit, assert_preparations_match,
)

if not ZIP_PATH.is_file():
    raise FileNotFoundError(f"Original Dataset V4 ZIP missing: {ZIP_PATH}")
print("Using shared source:", module_dir / "dataset_preparation.py")
print("Dataset ZIP:", ZIP_PATH)

Using shared source: /content/drive/MyDrive/MMA3001_v4_Project/src/dataset_preparation.py
Dataset ZIP: /content/drive/MyDrive/MMA3001_v4_Project/dataset/pork_rasher_v4.zip


In [11]:
# Clean overlapping sources and reserve the wrinkle source for validation.
manifest = prepare_dataset(
    zip_path=ZIP_PATH,
    output_dir=PREPARED_DIR,
    holdout_source="frame_with_red_8644_jpg",
    expected_holdout_images=3,
    expected_dataset_version=4,
    require_labels=True,
)
clean = PREPARED_DIR
print("Cleaned data.yaml:", clean / "data.yaml")
print("Cross-split overlaps removed:", manifest["removed_overlap_images"])
print("Wrinkle-source images moved into validation:", manifest["moved_to_validation"])
print_audit(manifest["audit"])

Cleaned data.yaml: /content/pork_rasher_dataset_clean/data.yaml
Cross-split overlaps removed: {'train': 36, 'valid': 1}
Wrinkle-source images moved into validation: 3

TRAIN: 3310 images; 684 backgrounds; 0 missing label files
  loose-meat: 66 boxes
  packaging-error: 1679 boxes
  twisted-meat: 72 boxes
  unsealed: 1052 boxes
  wrinkle: 15 boxes

VALID: 122 images; 14 backgrounds; 0 missing label files
  loose-meat: 3 boxes
  packaging-error: 34 boxes
  twisted-meat: 3 boxes
  unsealed: 80 boxes
  wrinkle: 3 boxes

TEST: 80 images; 12 backgrounds; 0 missing label files
  loose-meat: 3 boxes
  packaging-error: 23 boxes
  twisted-meat: 4 boxes
  unsealed: 49 boxes
  wrinkle: 0 boxes


In [12]:
# Compare image counts with the recorded baseline split.
EXPECTED_IMAGES = {"train": 3310, "valid": 122, "test": 80}
actual_images = {split: info["images"]
                 for split, info in manifest["audit"]["splits"].items()}
assert actual_images == EXPECTED_IMAGES, (
    f"Unexpected V4 split counts: {actual_images} versus {EXPECTED_IMAGES}. "
    "Inspect ZIP version and source grouping rather than adjusting labels."
)
assert manifest["audit"]["splits"]["test"]["boxes_by_class"]["wrinkle"] == 0
print("Historical split-count checks passed.")
print("Prepared split fingerprint:", manifest["split_fingerprint_sha256"])

# Compare the current split fingerprint with the saved record.
if not TRAINING_MANIFEST.exists():
    raise FileNotFoundError(
        f"Training manifest not found: {TRAINING_MANIFEST}. "
        "The reference manifest is required for split comparison."
    )
assert_preparations_match(manifest, TRAINING_MANIFEST)
print("Cleaned V4 splits match the saved preparation record.")

Historical split-count checks passed.
Prepared split fingerprint: 501d136cebd2cc8c5de2e2d4d050064f549ca88350f70a9a3ecf0cbbc38fdd69
Cleaned V4 splits match the saved preparation record.


In [13]:
# Load the saved baseline weights.
%pip install -q ultralytics
from ultralytics import YOLO

if not BASELINE_WEIGHTS.is_file():
    raise FileNotFoundError(f"Baseline model weights not found: {BASELINE_WEIGHTS}")
model = YOLO(str(BASELINE_WEIGHTS))

# Check class order against data.yaml.
model_names = model.names
if isinstance(model_names, dict):
    model_names = [model_names[k] for k in sorted(model_names)]
if list(model_names) != manifest["audit"]["class_names"]:
    raise ValueError("Model class order does not match V4 data.yaml class order")
print("Loaded weights:", BASELINE_WEIGHTS)
print("Class order verified:", model_names)

Loaded weights: /content/drive/MyDrive/MMA3001_v4_Project/model_runs/yolov8n_baseline/weights/best.pt
Class order verified: ['loose-meat', 'packaging-error', 'twisted-meat', 'unsealed', 'wrinkle']


In [14]:
# Evaluate the cleaned test split at 640 px with batch size 16.
test_results = model.val(
    data=str(clean / "data.yaml"),
    split="test",
    imgsz=640,
    batch=16,
    device=0,
    plots=True,
    project=str(RUNS_DIR),
    name="yolov8n_baseline_test",
)

print(f"Precision (mean): {test_results.box.mp:.4f}")
print(f"Recall (mean): {test_results.box.mr:.4f}")
print(f"mAP50: {test_results.box.map50:.4f}")
print(f"mAP50-95: {test_results.box.map:.4f}")
print("Evaluation plots stored at:", test_results.save_dir)
print("No labelled wrinkle boxes in this test split; wrinkle metrics are not established.")

Ultralytics 8.4.183 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 72 layers, 3,006,623 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1614.0±428.4 MB/s, size: 66.8 KB)
val: Scanning /content/pork_rasher_dataset_clean/test/labels... 80 images, 12 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 80/80 970.8it/s 0.1s
val: New cache created: /content/pork_rasher_dataset_clean/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 5/5 2.0it/s 2.5s
                   all         80         79      0.565       0.49      0.533      0.266
            loose-meat          2          3          1      0.606      0.731      0.329
       packaging-error         16         23      0.498      0.518      0.497      0.124
          twisted-meat          4          4          0          0     0.0612     0.0247
              unsealed         49         49       0.

## Results
Overall precision, recall, mAP50 and mAP50–95 describe the model's detection performance on the prepared V4 test split. Confusion-matrix and prediction plots provide further evidence of missed defects and incorrect detections.

The test split contains no labelled wrinkle examples. Wrinkle-specific recall and average precision therefore cannot be established from these test results.